# Step 1: Data Loading

In [2]:
import pandas as pd
import os

# Define paths
RAW_DATA_PATH = "../data/raw/"
CLEANED_DATA_PATH = "../data/cleaned/"

# Load datasets (assuming names are simplified)
df_income = pd.read_csv(os.path.join(RAW_DATA_PATH, "income_state.csv"))
df_cpi = pd.read_csv(os.path.join(RAW_DATA_PATH, "cpi_state.csv"))

# Quick look at the structure
print("Income Data Columns:", df_income.columns)
print("CPI Data Columns:", df_cpi.columns)

Income Data Columns: Index(['state', 'date', 'income_mean', 'income_median'], dtype='object')
CPI Data Columns: Index(['state', 'date', 'division', 'index'], dtype='object')


# Step 2: Date Preprocessing

In [3]:
# Convert to datetime
df_income['date'] = pd.to_datetime(df_income['date'])
df_cpi['date'] = pd.to_datetime(df_cpi['date'])

# Extract 'Year' to allow for merging later
df_income['year'] = df_income['date'].dt.year
df_cpi['year'] = df_cpi['date'].dt.year

# Step 3: Filter for Overall CPI only

In [4]:
# Without this, you are averaging 'Food', 'Transport', and 'Overall' together!
df_cpi_filtered = df_cpi[df_cpi['division'] == 'overall'].copy()

# Calculate average annual CPI per state
df_cpi_annual = df_cpi_filtered.groupby(['year', 'state'])['index'].mean().reset_index()
df_cpi_annual.rename(columns={'index': 'avg_annual_cpi'}, inplace=True)

print("CPI Aggregation Complete.")
df_cpi_annual.head()

CPI Aggregation Complete.


,year,state,avg_annual_cpi
0,2010,Johor,100.008333
1,2010,Kedah,100.000000
2,2010,Kelantan,100.008333
3,2010,Melaka,99.991667
4,2010,Negeri Sembilan,100.000000


# Step 4: The Merge

In [5]:
# We use 'income_median' because it matches your CSV headers
df_merged = pd.merge(df_income, df_cpi_annual, on=['year', 'state'], how='inner')

# Step 5: Calculate Real Median Income
# (Median Income / CPI) * 100 
df_merged['real_median_income'] = (df_merged['income_median'] / df_merged['avg_annual_cpi']) * 100

print(f"Merge complete! Final rows: {len(df_merged)}")
df_merged.head()

Merge complete! Final rows: 96


,state,date,income_mean,income_median,year,avg_annual_cpi,real_median_income
0,Johor,2012-01-01,4658,3650.0,2012,105.175000,3470.406465
1,Johor,2014-01-01,6207,5197.0,2014,111.641667,4655.072031
2,Johor,2016-01-01,6928,5652.0,2016,117.991667,4790.168797
3,Johor,2019-01-01,8013,6427.0,2019,125.083333,5138.174550
4,Johor,2020-01-01,7264,5690.0,2020,123.258333,4616.320736


In [6]:
df_merged.to_csv(os.path.join(CLEANED_DATA_PATH, "malaysia_living_costs_cleaned.csv"), index=False)
print("Cleaned data saved to data/cleaned/")

Cleaned data saved to data/cleaned/
